# Probability & Statistics Lab 1: Naive Bayes Classifier

Work breakdown:

* *Ihor Kastranets* (3, 4);
* *Tsiuk Sofia* (5);
* *Yulia Yakymiv* (1, 2).

## Introduction

During the first three weeks, you learned a couple of essential notions
and theorems, and one of the most important among them is the **Bayes
theorem**:

$$
\mathsf P(A \mid B) = \frac{\mathsf P(A)\mathsf P(B \mid A)}{\mathsf P(B)}.
$$

**Naive Bayes Classifier** is a simple algorithm, which is based on
Bayes theorem and used for solving classification problems.

**Classification problem** is a problem in which an observation has to
be classified in one of the $n$ classes based on its similarity with
observations in each class.

It is a **probabilistic classifier**, which means it predicts based on
the probability of an observation belonging to each class. To compute
it, this algorithm uses **Bayes' formula** as:
$$
\mathsf P (\text{class}_i \mid \text{observation}) = \frac{\mathsf P(\text{class}_i) \mathsf P(\text{observation} \mid \text{class}_i)}{\mathsf P(\text{observation})}. \quad \quad (1)
$$
Breaking down the pieces of the formula,

* $\mathsf P(\text{class}_i)$ is referred to as a **prior** [distribution]: it provides initial class probabilities, influenced by prior beliefs about the problem.
* $\mathsf P(\text{observation} \mid \text{class}_i)$ is the **likelihood**: it tells how likely observations are for a chosen class.
* $\mathsf P(\text{observation})$ is the **evidence**; usually isn't computed directly and used as a normalization constant s.t. sum of $\mathsf P (\text{class}_i \mid \text{observation})$ for all classes is $1$.

This lab focuses on binary classification of text. Text can be considered a sequence of observations/features, with many ways to split it into individual/elementary features. Here, we split the text by words, and one word is considered a feature $\text{feature}_j$:
$\mathsf P(\text{text}) = \mathsf P(\text{features}_1, \text{feature}_2, \dots, \text{feature}_N) = \mathsf P(\text{features}_1)\cdot \mathsf P(\text{feature}_2 \mid \text{feature}_1) \cdot \mathsf P(\text{feature}_3 \mid (\text{feature}_1, \text{feature}_2)) \dots.$

However, modeling such long conditionals is complex, compute-intensive, and requires lots of data.

One way to deal with this is to assume independence, where one can write: $\mathsf P(\text{text}) = \prod_{j=1}^N \mathsf P(\text{feature}_j).$
Or, one could assume the dependence only on the previous word, which will be explored in the additional task!

$\mathsf P(\text{feature}_j)$ is unknown, and we estimate it empirically using the training data split. The empirical probability of some feature $x$ would be $\frac{\text{\# of } x}{\text{total \# of features}}.$
Note that such an approach would turn $\mathsf P(\text{text})$ to $0$ for any completely new feature (as its count among data is zero), which is undesirable.
A common way to deal with this is [Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing). No need to dive deeply into this now, as you will develop a better intuition as you progress through the course.
In this lab, you shall apply Laplace smoothing to both class priors and feature ($\log$–)probabilities.

#### Implementation note

Instead of storing and calculating probabilities directly, many ML solutions prefer computations in $\log$ domain. One of the reasons is numerical stability, which shows up in $\prod_{j=1}^N \mathsf P(\text{feature}_j)$:
each $\mathsf P(\text{feature}_j) \in (0, 1)$ in practice, resulting in tiny final product, which floating-point numbers the computer uses can't store as accurately.
However, in the logarithmic domain even $10^{-100}$ corresponds to an order of magnitude of $-100$.

So, let's rewrite the equation we estimate:
$$
\log [\mathsf P (\text{class}_i \mid \text{observation})] = \underbrace{\log[\mathsf P(\text{class}_i)] + \sum_{j=1}^N \log[\mathsf P(\text{feature}_j)]}_{\text{used to be a numerator, } L_i} - \overbrace{\log[\mathsf P(\text{observation})]}^{\text{used to be a denominator, log-evidence}}. \quad \quad (2)
$$

[A video example of a Naive Bayes](https://www.youtube.com/watch?v=O2L2Uv9pdDA).

### Outline of the work

1. **Data preprocessing** (1 point)
2. **Data visualization** (1 point)
3. **Classifier implementation from scratch** (1.5 points)
4. **Measurements of effectiveness of your classifier** (1.5 points)
5. **Additional task** (up to 2 points, optional)

*Please submit a pre-executed `.ipynb` notebook for easier evaluation.*

### Data preprocessing

In [ ]:
import pathlib
import re
import math
from collections import defaultdict
import pandas as pd
import numpy as np


class Dataset:
    def __init__(self, dataset_dir_path: pathlib.Path):
        self.train_df = pd.read_csv(dataset_dir_path / "train.csv")
        self.test_df = pd.read_csv(dataset_dir_path / "test.csv")

        self.bag_of_words_per_label_id: dict[int, dict[str, int]] = {}
        self.bag_of_log_probabilities_per_label_id: dict[int, dict[str, float]] = {}

    def preprocess_text(self, stop_words: list[str]) -> None:
        """
        Adds "label-id" (starting with 0) and "useful-words" columns to dataframes.
        Useful words are the ones that aren't in the stop words list.
        Remove the punctuation, map all the text to one register (e.g., lowercase).

        You may add custom preprocessing if it's justified!
        """
        stop_words = {word for line in stop_words for word in line.split()}
        label_to_id = {"neutral": 0, "discrim": 1}

        def clean(text: str) -> list[str]:
            text = text.lower()                                # one register
            text = text.replace("â\u0080\u0099", "'")     # broken-encoding right apostrophe: donâ€™t -> don't
            text = re.sub(r"&\w+;", " ", text)                 # HTML entities: &amp; &gt; &lt;
            text = text.replace("@user", " ")                  # anonymized username placeholder
            words = re.findall(r"[a-z']+", text)               # keep only English letters and apostrophes
            words = [w.strip("'") for w in words]              # apostrophes at the edges of a word
            words = [w for w in words if w not in stop_words]  # stop words (some contain an apostrophe: don't, it's)
            words = [w.replace("'", "") for w in words]        # trump's -> trumps
            return [w for w in words if len(w) >= 2]           # drop single letters: s, t, m, u

        for df in [self.train_df, self.test_df]:
            df["label-id"] = df["label"].map(label_to_id)
            df["useful-words"] = df["tweet"].apply(clean)


    def calculate_classes_prior(self, smoothing_alpha: float) -> np.ndarray:
        """
        :param smoothing_alpha: parameter of Laplace smoothing.
        :return: distribution of classes after smoothing, starting with label-id 0.
        """
        class_counts = self.train_df["label-id"].value_counts().sort_index().to_numpy()
        number_of_classes = len(class_counts)
        total = class_counts.sum()
        return (class_counts + smoothing_alpha) / (total + smoothing_alpha * number_of_classes)


    def compute_bag_of_words(self) -> None:
        """
        Counts the number of occurrences of each word per label id and updates `self.bag_of_words_per_label_id`.

        Bag of words intro: https://machinelearningmastery.com/gentle-introduction-bag-words-model/.
        Advice: you may use `collections.defaultdict` (https://docs.python.org/3/library/collections.html#collections.defaultdict).
        """
        self.bag_of_words_per_label_id = {}
        for label_id, words in zip(self.train_df["label-id"], self.train_df["useful-words"]):
            if label_id not in self.bag_of_words_per_label_id:
                self.bag_of_words_per_label_id[label_id] = defaultdict(int)
            for word in words:
                self.bag_of_words_per_label_id[label_id][word] += 1


    def compute_bag_of_log_probabilities(self, smoothing_alpha: float) -> float:
        """
        Uses `self.bag_of_words_per_label_id` and updates `self.bag_of_log_probabilities_per_label_id`.
        `self.bag_of_log_probabilities_per_label_id` should store log[P(feature_j)].

        :param smoothing_alpha: parameter of Laplace smoothing.
        :return the log-probability of a new(previously unseen) feature, necessary for evaluation on the test set.
        """
        vocabulary = set()
        for bag in self.bag_of_words_per_label_id.values():
            vocabulary.update(bag.keys())
        vocabulary_size = len(vocabulary)

        for label_id, bag in self.bag_of_words_per_label_id.items():
            words_in_class = sum(bag.values())
            denominator = words_in_class + smoothing_alpha * vocabulary_size
            self.bag_of_log_probabilities_per_label_id[label_id] = {}
            for word in vocabulary:
                self.bag_of_log_probabilities_per_label_id[label_id][word] = math.log(
                    (bag.get(word, 0) + smoothing_alpha) / denominator
                )

        total_words = sum(sum(bag.values()) for bag in self.bag_of_words_per_label_id.values())
        return math.log(smoothing_alpha / (total_words + smoothing_alpha * vocabulary_size))



discrimination_dataset_path = pathlib.Path("discrimination")  # change if needed.
discrimination_dataset = Dataset(discrimination_dataset_path)

labels = discrimination_dataset.train_df["label"].unique().tolist()
print(f"Labels are {labels}.")

Loading the stop words

In [ ]:
stop_words_path = pathlib.Path("stop_words.txt")  # change if needed.

stop_words: list[str] = []
with open(stop_words_path) as f:
    for line in f:
        stop_words.append(line.strip())

print(f"Some of the stop words are {stop_words[:8]}.")

In [ ]:
discrimination_dataset.preprocess_text(stop_words)

In [ ]:
classes_alpha = 1
discrimination_dataset.calculate_classes_prior(classes_alpha)

In [ ]:
discrimination_dataset.compute_bag_of_words()

In [ ]:
features_alpha = 1
unseen_feature_log_prob = discrimination_dataset.compute_bag_of_log_probabilities(features_alpha)

### Data visualization

Visualize the dataset, its properties, etc. You may use external libraries such as `matplotlib`, `plotly`, etc. Data analysis could provide hints/ideas for downstream classifier evaluation.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

train_df = discrimination_dataset.train_df
test_df = discrimination_dataset.test_df

colors = {"neutral": "#2a78d6", "discrim": "#eb6834"}
class_names = ["neutral", "discrim"]

# Number of tweets in each class
train_counts = train_df["label"].value_counts()
test_counts = test_df["label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in [(axes[0], train_counts, "Train"), (axes[1], test_counts, "Test")]:
    values = [counts[name] for name in class_names]
    bars = ax.bar(class_names, values, color=[colors[name] for name in class_names])
    total = sum(values)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value} ({value / total:.1%})",
                ha="center", va="bottom")
    ax.set_title(f"{title}: number of tweets per class")
    ax.set_ylabel("number of tweets")
plt.tight_layout()
plt.show()

# Tweet length (number of words after preprocessing)
train_df["n_words"] = train_df["useful-words"].apply(len)

plt.figure(figsize=(8, 4))
for name in class_names:
    lengths = train_df[train_df["label"] == name]["n_words"]
    # density=True shows shares instead of counts, so the small class is comparable
    plt.hist(lengths, bins=range(0, 30), alpha=0.6, density=True,
             label=name, color=colors[name])
    print(f"{name}: average number of words after preprocessing = {lengths.mean():.1f}")
plt.title("Tweet length after preprocessing (train)")
plt.xlabel("number of words after preprocessing")
plt.ylabel("share of tweets")
plt.legend()
plt.show()

# Most frequent words in each class
word_counts = {}
for name in class_names:
    counter = Counter()
    for words in train_df[train_df["label"] == name]["useful-words"]:
        counter.update(words)
    word_counts[name] = counter

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name in zip(axes, class_names):
    top = word_counts[name].most_common(20)
    words = [w for w, c in top][::-1]      # reversed so the most frequent word is on top
    counts = [c for w, c in top][::-1]
    ax.barh(words, counts, color=colors[name])
    ax.set_title(f"Top-20 words in class '{name}'")
    ax.set_xlabel("number of occurrences")
plt.tight_layout()
plt.show()

# How many test words never appear in train
train_vocab = set(word_counts["neutral"]) | set(word_counts["discrim"])
test_words = [w for words in test_df["useful-words"] for w in words]
unseen = [w for w in test_words if w not in train_vocab]
print(f"Unique words in train: {len(train_vocab)}")
print(f"Words in test that never appear in train: {len(unseen)} of {len(test_words)} "
      f"({len(unseen) / len(test_words):.1%})")

### Classifier implementation

Recall equation $(2)$. You will manually calculate the *"used to be a numerator"* part, call it $L_i$, for each class, while log-evidence is numerically safely calculated from an array of log-domain per-class numerators.

In [ ]:
def compute_log_evidence(per_label_log_domain_numerator: np.ndarray) -> float:
    return np.logaddexp.reduce(per_label_log_domain_numerator)

#### Derivation of log-evidence calculation

Assume all $L_i$ are calculated. Derive how one would calculate log-evidence. Later explain how it connects to the provided above `compute_log_evidence` function.

Hint: use the property of valid probability distributions.

---

### Derivation

Recall equation $(2)$ for the posterior probability of class $i$ given the observation:
$$
\log [\mathsf P (\text{class}_i \mid \text{observation})] = L_i - \log[\mathsf P(\text{observation})]
$$
where $L_i$ is the log-domain numerator:
$$
L_i = \log[\mathsf P(\text{class}_i)] + \sum_{j=1}^N \log[\mathsf P(\text{feature}_j \mid \text{class}_i)]
$$

Exponentiating both sides gives the posterior probability:
$$
\mathsf P (\text{class}_i \mid \text{observation}) = \exp\Big(L_i - \log[\mathsf P(\text{observation})]\Big) = \frac{\exp(L_i)}{\mathsf P(\text{observation})}
$$

By the **property of a valid probability distribution**, the sum of the posterior probabilities over all mutually exclusive and exhaustive classes $i \in \{1, \dots, C\}$ must equal $1$:
$$
\sum_{i=1}^C \mathsf P(\text{class}_i \mid \text{observation}) = 1
$$

Substituting the posterior expression into the summation:
$$
\sum_{i=1}^C \frac{\exp(L_i)}{\mathsf P(\text{observation})} = 1
$$

Since the evidence $\mathsf P(\text{observation})$ is independent of the class index $i$, it can be factored out of the summation:
$$
\frac{1}{\mathsf P(\text{observation})} \sum_{i=1}^C \exp(L_i) = 1 \implies \mathsf P(\text{observation}) = \sum_{i=1}^C \exp(L_i)
$$

Now, taking the natural logarithm of both sides yields the **log-evidence**:
$$
\log[\mathsf P(\text{observation})] = \log \left( \sum_{i=1}^C \exp(L_i) \right)
$$

---

### Connection to `compute_log_evidence`

In the code cell above, the function is implemented as:

```python
def compute_log_evidence(per_label_log_domain_numerator: np.ndarray) -> float:
    return np.logaddexp.reduce(per_label_log_domain_numerator)
```

`np.logaddexp(x1, x2)` calculates the log of the sum of exponentiations for two numbers:
$$
\text{logaddexp}(x_1, x_2) = \log\big(\exp(x_1) + \exp(x_2)\big)
$$
The `.reduce()` method applies this binary operation cumulatively across all elements of the 1D array of per-class numerators $[L_1, L_2, \dots, L_C]$:
$$
\text{np.logaddexp.reduce}([L_1, L_2, \dots, L_C]) = \log \left( \sum_{i=1}^C \exp(L_i) \right)
$$
which is identical to the derived log-evidence formula.

Computing $\sum_{i=1}^C \exp(L_i)$ directly by first exponentiating $L_i$ and then summing often causes **arithmetic overflow** (if $L_i \gg 0$) or **underflow to zero** (if $L_i \ll 0$).
`np.logaddexp` overcomes this by internally applying the numerically stable identity:
$$
\log\big(\exp(x_1) + \exp(x_2)\big) = \max(x_1, x_2) + \log\big(1 + \exp(-|x_1 - x_2|)\big)
$$
This prevents overflow and maintains precision in floating-point calculations throughout model predictions.

#### Prediction implementation

In [ ]:
from typing import Literal


def predict(dataset: Dataset, split: Literal["train", "test"]) -> np.ndarray:
    """
    "train" split option may be useful for debugging.
    :return: the (NxC) array of predicted probabilities, where N is the number of texts, and C is the number of classes.
    """
    if split == "train":
        df = dataset.train_df
    else:
        df = dataset.test_df

    log_priors = np.log(dataset.calculate_classes_prior(classes_alpha))

    predicted_probabilities = []
    for words in df["useful-words"]:
        numerators = []
        for label_id in range(len(log_priors)):
            word_log_probs = dataset.bag_of_log_probabilities_per_label_id[label_id]
            total = log_priors[label_id]
            for word in words:
                total += word_log_probs.get(word, unseen_feature_log_prob)
            numerators.append(total)
        numerators = np.array(numerators)
        log_evidence = compute_log_evidence(numerators)
        predicted_probabilities.append(np.exp(numerators - log_evidence))

    return np.array(predicted_probabilities)

### Classifier evaluation

Note that accuracy is not always a good metric for the classifier. One may look at precision and recall curves, F1 score metric, ROC curves.

When evaluating the model, it's important to understand the
different types of classification results:

* A ***true positive***(TP) result is one where the model correctly
  predicts the positive class.
* A ***true negative***(TN) result is one where the model correctly
  predicts the negative class.
* A ***false positive***(FP) result is one where the model incorrectly
  predicts the positive class when it is actually negative.
* A ***false negative***(FN) result is one where the model incorrectly
  predicts the negative class when it is actually positive.

Precision measures the proportion of true positive predictions among all positive predictions made by the model: $\text{Precision} = \frac{TP}{TP+FP}.$

Recall measures the proportion of true positives identified out of all actual positive cases: $\text{Recall} = \frac{TP}{TP+FN}$.

F1 score is the harmonic mean of both precision and recall: $\text{F1} = \frac{2\times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}.$

See [this link](https://cohere.com/blog/classification-eval-metrics) to find more information about metrics.

Visualize plots and show failure cases.

In [ ]:
def count_metrics(real, predicted):
    tp = int(((predicted == 1) & (real == 1)).sum())
    tn = int(((predicted == 0) & (real == 0)).sum())
    fp = int(((predicted == 1) & (real == 0)).sum())
    fn = int(((predicted == 0) & (real == 1)).sum())
    accuracy = (tp + tn) / len(real)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    return tp, tn, fp, fn, accuracy, precision, recall, f1


probabilities = predict(discrimination_dataset, "test")
predicted = probabilities.argmax(axis=1)
real = test_df["label-id"].to_numpy()

tp, tn, fp, fn, accuracy, precision, recall, f1 = count_metrics(real, predicted)

print(f"TP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print(f"Accuracy  = {accuracy:.3f}")
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1        = {f1:.3f}")

always_neutral_accuracy = (real == 0).mean()
print(f"Accuracy of a model that always says 'neutral' = {always_neutral_accuracy:.3f}")

In [ ]:
matrix = np.array([[tn, fp], [fn, tp]])

plt.figure(figsize=(4, 4))
plt.imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, matrix[i, j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], class_names)
plt.yticks([0, 1], class_names)
plt.xlabel("predicted")
plt.ylabel("real")
plt.title("Confusion matrix")
plt.show()

thresholds = np.linspace(0.01, 0.99, 99)
precisions = []
recalls = []
f1_scores = []
for threshold in thresholds:
    result = count_metrics(real, (probabilities[:, 1] >= threshold).astype(int))
    precisions.append(result[5])
    recalls.append(result[6])
    f1_scores.append(result[7])

plt.figure(figsize=(8, 4))
plt.plot(thresholds, precisions, label="precision")
plt.plot(thresholds, recalls, label="recall")
plt.plot(thresholds, f1_scores, label="F1")
plt.xlabel("threshold for class 'discrim'")
plt.ylabel("metric value")
plt.title("Metrics for different thresholds")
plt.legend()
plt.show()

best = int(np.argmax(f1_scores))
print(f"Best F1 = {f1_scores[best]:.3f} at threshold {thresholds[best]:.2f}")

In [ ]:
fp_indices = np.where((predicted == 1) & (real == 0))[0]
fn_indices = np.where((predicted == 0) & (real == 1))[0]

worst_fp = fp_indices[np.argsort(-probabilities[fp_indices, 1])][:5]
worst_fn = fn_indices[np.argsort(probabilities[fn_indices, 1])][:5]

print("TOP FALSE POSITIVES (Actual: Neutral, Model predicted: Discrim)\n")
for idx in worst_fp:
    print(f"Index: {idx} | P(discrim) = {probabilities[idx, 1]:.4f}")
    print(f"Tweet: {discrimination_dataset.test_df['tweet'].iloc[idx]}")
    print(f"Words: {discrimination_dataset.test_df['useful-words'].iloc[idx]}")
    print("-" * 80)

print("TOP FALSE NEGATIVES (Actual: Discrim, Model predicted: Neutral)\n")
for idx in worst_fn:
    print(f"Index: {idx} | P(discrim) = {probabilities[idx, 1]:.4f}")
    print(f"Tweet: {discrimination_dataset.test_df['tweet'].iloc[idx]}")
    print(f"Words: {discrimination_dataset.test_df['useful-words'].iloc[idx]}")
    print("-" * 80)

### Additional task

You are asked to explore, implement, evaluate, and analyze another way to calculate $\mathsf P(\text{text})$; mainly, by conditioning on the previous feature:
$$
\mathsf P(\text{text}) = \prod_{j=2}^N \mathsf P(\text{feature}_j \mid \text{feature}_{j-1}).
$$

The task will be graded proportionally to analysis depth and conclusions viability.

In [ ]:
class BigramNaiveBayes:
    def __init__(self, dataset, alpha=1.0):
        self.dataset = dataset
        self.alpha = alpha
        self.bigram_counts = {}
        self.unigram_counts = {}
        self.vocabulary = set()

    def train(self):
        for label_id in self.dataset.bag_of_words_per_label_id.keys():
            self.bigram_counts[label_id] = defaultdict(int)
            self.unigram_counts[label_id] = defaultdict(int)

        for label_id, words in zip(self.dataset.train_df["label-id"], self.dataset.train_df["useful-words"]):
            if len(words) < 2:
                continue

            for i in range(1, len(words)):
                w_prev, w_curr = words[i-1], words[i]
                self.bigram_counts[label_id][(w_prev, w_curr)] += 1
                self.unigram_counts[label_id][w_prev] += 1
                self.vocabulary.add(w_curr)
                self.vocabulary.add(w_prev)

        self.vocab_size = len(self.vocabulary)
        self.log_priors = np.log(self.dataset.calculate_classes_prior(self.alpha))

    def get_log_prob(self, label_id, w_prev, w_curr):
        bigram_count = self.bigram_counts[label_id].get((w_prev, w_curr), 0)
        unigram_count = self.unigram_counts[label_id].get(w_prev, 0)

        numerator = bigram_count + self.alpha
        denominator = unigram_count + self.alpha * self.vocab_size
        return math.log(numerator / denominator)

    def predict(self, split="test"):
        df = self.dataset.train_df if split == "train" else self.dataset.test_df
        predicted_probabilities = []

        for words in df["useful-words"]:
            numerators = []
            for label_id in range(len(self.log_priors)):
                total_log_prob = self.log_priors[label_id]

                if len(words) >= 2:
                    for i in range(1, len(words)):
                        total_log_prob += self.get_log_prob(label_id, words[i-1], words[i])
                else:
                    total_log_prob += math.log(1.0 / self.vocab_size)

                numerators.append(total_log_prob)

            numerators = np.array(numerators)
            log_evidence = np.logaddexp.reduce(numerators)
            predicted_probabilities.append(np.exp(numerators - log_evidence))

        return np.array(predicted_probabilities)

bigram_model = BigramNaiveBayes(discrimination_dataset, alpha=1.0)
bigram_model.train()
bigram_probs = bigram_model.predict("test")
bigram_predicted = bigram_probs.argmax(axis=1)

tp, tn, fp, fn, accuracy, precision, recall, f1 = count_metrics(real, bigram_predicted)
print("Bigram Model Evaluation")
print(f"TP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")
print(f"Accuracy  = {accuracy:.3f}")
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1        = {f1:.3f}")

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
error_rates = []
fp_rates = []
fn_rates = []

total_samples = len(real)

for threshold in thresholds:
    predicted = (probabilities[:, 1] >= threshold).astype(int)

    fp = ((predicted == 1) & (real == 0)).sum()
    fn = ((predicted == 0) & (real == 1)).sum()

    error_rates.append((fp + fn) / total_samples)
    fp_rates.append(fp / total_samples)
    fn_rates.append(fn / total_samples)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, error_rates, label="Error Rate", color="#d62728", linewidth=2.5)
plt.plot(thresholds, fp_rates, label="False Positives", color="#ff7f0e", linestyle="--")
plt.plot(thresholds, fn_rates, label="False Negatives", color="#1f77b4", linestyle="--")

plt.xlabel("Threshold for class 'discrim'", fontsize=12)
plt.ylabel("Error Fraction", fontsize=12)
plt.title("Error Rate vs Threshold", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()

### Conclusions

* Method and Mathematical Foundations: The implemented Naive Bayes Classifier relies on Bayes' theorem to predict the probability of an observation belonging to a specific class. The standard unigram model makes a "naive" assumption of conditional independence, meaning it calculates the likelihood of a text by multiplying the probabilities of individual features (words). To prevent numerical underflow, these calculations are performed in the log-domain. We also explored a bigram model based on the Markov assumption, where a word's probability is conditioned on the immediately preceding word. Laplace smoothing was applied in both cases to prevent zero probabilities for unseen words.
* Pros, Cons, and Assumptions: The primary advantage of Naive Bayes is its computational efficiency, simplicity, and speed. However, a major limitation is its core assumption that features (words) are completely independent, which ignores sentence structure, negations, and context. When attempting to capture context using the bigram method, the model suffers from severe data sparsity, as a vast number of word pairs in the test set never appeared during training.
* Accuracy vs. F1-Score: Accuracy is a highly misleading metric for classification tasks involving imbalanced data. Because 93% of the dataset belongs to the "neutral" class, a model that simply predicts "neutral" for every single text would achieve an accuracy of 0.930 while completely failing to detect any discriminatory tweets. The F1 score is generally preferable because it is the harmonic mean of Precision and Recall, providing a much more objective measure of the model's ability to accurately identify the minority positive class (discriminatory tweets).